In [4]:
import os
import shutil

SOURCE_DIR = "/kaggle/input/datasets/sprakashbaranwal/saved-pre-trained-model-zip/saved_pre_trained_model"
DEST_DIR = "/kaggle/working/saved_pre_trained_model"

# Remove existing destination if present
if os.path.exists(DEST_DIR):
    shutil.rmtree(DEST_DIR)

# Copy the pre-trained model to /kaggle/working
shutil.copytree(SOURCE_DIR, DEST_DIR)

print("=" * 70)
print("[Model Setup] Pre-trained model copied successfully")
print(f"[Source]      {SOURCE_DIR}")
print(f"[Destination] {DEST_DIR}")
print("=" * 70)

# Verify copied files
for root, dirs, files in os.walk(DEST_DIR):
    for file in files:
        print(os.path.join(root, file))

[Model Setup] Pre-trained model copied successfully
[Source]      /kaggle/input/datasets/sprakashbaranwal/saved-pre-trained-model-zip/saved_pre_trained_model
[Destination] /kaggle/working/saved_pre_trained_model
/kaggle/working/saved_pre_trained_model/generation_config.json
/kaggle/working/saved_pre_trained_model/tokenizer_config.json
/kaggle/working/saved_pre_trained_model/config.json
/kaggle/working/saved_pre_trained_model/model.safetensors
/kaggle/working/saved_pre_trained_model/tokenizer.json


In [5]:
"""
===============================================================================
File Name  : train.py

Purpose:
    Fine-tuning pipeline for T5-small Encoder-Decoder model on SQuAD v1.1.

    Final experiment configuration:
        - Dataset samples : 20,000
        - Train split     : 80% = 16,000
        - Validation      : 10% = 2,000
        - Test            : 10% = 2,000
        - Epochs          : 3
        - Optimizer       : AdamW
        - Learning rate   : 5e-5
        - Warmup          : 10% of total training steps
        - Batch size      : 16 on CUDA
        - Input length    : 512 tokens
        - Target length   : 64 tokens
        - FP16            : Enabled on CUDA

Generated artifacts:
        - saved_model/
        - training_log.csv
        - loss_curve.png
        - test_results.json

Course  : Natural Language Processing (S2-25_AIMLCZG530)
Program : M.Tech. in AIML, BITS Pilani (WILP)
Group   : Group 238
===============================================================================
"""

# =============================================================================
# 1. IMPORTS
# =============================================================================

import sys
import os
import time
import json
from pathlib import Path

import torch
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    TrainerCallback
)


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================

SEED = 42

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 3. PATH DEFINITIONS
# =============================================================================

# Works when executed as:
#     python train.py
#
# Also works when executed inside Jupyter/Kaggle:
#     %run train.py

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()


# Local pre-trained T5 model uploaded to Kaggle
BASE_MODEL = "/kaggle/working/saved_pre_trained_model"

# Output artifacts
SAVED_MODEL_DIR = BASE_DIR / "saved_model"
CSV_LOG_PATH = BASE_DIR / "training_log.csv"
PLOT_IMAGE_PATH = BASE_DIR / "loss_curve.png"
TEST_METRICS_PATH = BASE_DIR / "test_results.json"


# =============================================================================
# 4. FINAL EXPERIMENT CONFIGURATION
# =============================================================================

DEFAULT_NUM_SAMPLES = 20000
DEFAULT_BATCH_SIZE = 16

NUM_EPOCHS = 3

LEARNING_RATE = 5e-5

WARMUP_RATIO = 0.10

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64

LOGGING_STEPS = 50
EVAL_STEPS = 50


# =============================================================================
# 5. DATASET PREPARATION
# =============================================================================

def prepare_squad_data(tokenizer, num_samples=DEFAULT_NUM_SAMPLES):
    """
    Load SQuAD v1.1, highlight the target answer span using <hl> tags,
    tokenize the input and target question, and create an 80/10/10 split.

    For 20,000 samples:
        Train      = 16,000
        Validation = 2,000
        Test       = 2,000
    """

    print(
        f"[1/4] Loading SQuAD v1.1 "
        f"(sample subset size: {num_samples})...",
        flush=True
    )

    raw_dataset = load_dataset(
        "rajpurkar/squad",
        split=f"train[:{num_samples}]"
    )

    # -------------------------------------------------------------------------
    # Preprocessing
    # -------------------------------------------------------------------------

    def preprocess_function(examples):

        inputs = []
        targets = []

        for context, answer, question in zip(
            examples["context"],
            examples["answers"],
            examples["question"]
        ):

            # SQuAD answer text
            ans_text = answer["text"][0]

            # Highlight the first occurrence of the answer
            if ans_text in context:

                highlighted_context = context.replace(
                    ans_text,
                    f"<hl> {ans_text} <hl>",
                    1
                )

                input_text = (
                    f"generate question: context: "
                    f"{highlighted_context}"
                )

            else:

                # Fallback if answer text is not found
                input_text = (
                    f"generate question: context: {context}"
                )

            inputs.append(input_text)
            targets.append(question)

        # ---------------------------------------------------------------------
        # Tokenize input/context
        # ---------------------------------------------------------------------

        model_inputs = tokenizer(
            inputs,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding="max_length"
        )

        # ---------------------------------------------------------------------
        # Tokenize target questions
        # ---------------------------------------------------------------------

        labels = tokenizer(
            targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length"
        )

        # ---------------------------------------------------------------------
        # Ignore padding tokens during loss calculation
        # ---------------------------------------------------------------------

        labels_ids = [
            [
                token if token != tokenizer.pad_token_id else -100
                for token in label
            ]
            for label in labels["input_ids"]
        ]

        model_inputs["labels"] = labels_ids

        return model_inputs

    # -------------------------------------------------------------------------
    # Apply preprocessing
    # -------------------------------------------------------------------------

    processed_dataset = raw_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=raw_dataset.column_names
    )

    # -------------------------------------------------------------------------
    # 80% Train / 20% Temporary
    # -------------------------------------------------------------------------

    train_test_split = processed_dataset.train_test_split(
        test_size=0.20,
        seed=SEED
    )

    # -------------------------------------------------------------------------
    # Split temporary 20% equally:
    # 10% Validation / 10% Test
    # -------------------------------------------------------------------------

    val_test_split = train_test_split["test"].train_test_split(
        test_size=0.50,
        seed=SEED
    )

    split_dataset = DatasetDict({
        "train": train_test_split["train"],
        "validation": val_test_split["train"],
        "test": val_test_split["test"]
    })

    print(
        f"[Dataset Split] "
        f"Train: {len(split_dataset['train'])} | "
        f"Validation: {len(split_dataset['validation'])} | "
        f"Test: {len(split_dataset['test'])}",
        flush=True
    )

    return split_dataset


# =============================================================================
# 6. TRAINING / VALIDATION LOSS LOGGER
# =============================================================================

class TrainingLoggerCallback(TrainerCallback):
    """
    Captures training loss and validation loss during training.

    Results are saved to:
        training_log.csv

    A combined training/validation loss graph is saved to:
        loss_curve.png
    """

    def __init__(self):

        self.train_history = []
        self.eval_history = []

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):

        if not logs:
            return

        epoch_val = (
            round(state.epoch, 2)
            if state.epoch is not None
            else 0.0
        )

        # ---------------------------------------------------------------------
        # Training loss
        # ---------------------------------------------------------------------

        if "loss" in logs:

            loss_val = round(float(logs["loss"]), 4)

            self.train_history.append({
                "step": state.global_step,
                "epoch": epoch_val,
                "train_loss": loss_val
            })

            print(
                f"[Training Step {state.global_step}] "
                f"Epoch {epoch_val} | "
                f"Training Loss: {loss_val}",
                flush=True
            )

        # ---------------------------------------------------------------------
        # Validation loss
        # ---------------------------------------------------------------------

        if "eval_loss" in logs:

            eval_loss_val = round(
                float(logs["eval_loss"]),
                4
            )

            self.eval_history.append({
                "step": state.global_step,
                "epoch": epoch_val,
                "eval_loss": eval_loss_val
            })

            print(
                f"[Validation Step {state.global_step}] "
                f"Epoch {epoch_val} | "
                f"Validation Loss: {eval_loss_val}",
                flush=True
            )

    def export_artifacts(self):

        df_train = pd.DataFrame(self.train_history)
        df_eval = pd.DataFrame(self.eval_history)

        # ---------------------------------------------------------------------
        # Create combined CSV
        # ---------------------------------------------------------------------

        if not df_train.empty and not df_eval.empty:

            df_combined = pd.merge(
                df_train,
                df_eval,
                on=["step", "epoch"],
                how="outer"
            ).sort_values("step")

        elif not df_train.empty:

            df_combined = df_train

        else:

            df_combined = df_eval

        df_combined.to_csv(
            str(CSV_LOG_PATH),
            index=False
        )

        print(
            f"[Artifact] Unified training/validation metrics "
            f"log saved to '{CSV_LOG_PATH.name}'",
            flush=True
        )

        # ---------------------------------------------------------------------
        # Generate loss curve
        # ---------------------------------------------------------------------

        plt.figure(figsize=(9, 5))

        if not df_train.empty:

            plt.plot(
                df_train["step"],
                df_train["train_loss"],
                marker="o",
                linewidth=2,
                label="Training Loss"
            )

        if not df_eval.empty:

            plt.plot(
                df_eval["step"],
                df_eval["eval_loss"],
                marker="s",
                linewidth=2,
                linestyle="--",
                label="Validation Loss"
            )

        plt.title(
            "Step-Level Training and Validation Loss "
            "(T5-Small on SQuAD v1.1)"
        )

        plt.xlabel(
            "Optimization Step "
            "(Logged Every 50 Steps)"
        )

        plt.ylabel("Cross-Entropy Loss")

        plt.grid(
            True,
            linestyle="--",
            alpha=0.6
        )

        plt.legend()

        plt.tight_layout()

        plt.savefig(
            str(PLOT_IMAGE_PATH),
            dpi=300
        )

        plt.close()

        print(
            f"[Artifact] Step-level loss graph exported to "
            f"'{PLOT_IMAGE_PATH.name}'",
            flush=True
        )


# =============================================================================
# 7. DEVICE DETECTION
# =============================================================================

def detect_device():
    """
    Detect CUDA GPU, Apple MPS GPU, or CPU.

    CUDA:
        Batch size = 16
        FP16 = True

    MPS / CPU:
        Batch size = 8
        FP16 = False
    """

    if torch.cuda.is_available():

        device_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()

        return (
            "cuda",
            f"CUDA GPU ({device_name}) x{gpu_count}",
            True,
            16
        )

    elif (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ):

        return (
            "mps",
            "Apple Silicon MPS (GPU)",
            False,
            8
        )

    else:

        return (
            "cpu",
            "System CPU",
            False,
            8
        )


# =============================================================================
# 8. MAIN TRAINING PIPELINE
# =============================================================================

def main():

    start_time = time.time()

    # -------------------------------------------------------------------------
    # Runtime configuration
    # -------------------------------------------------------------------------

    num_samples = int(
        os.environ.get(
            "NUM_SAMPLES",
            str(DEFAULT_NUM_SAMPLES)
        )
    )

    requested_batch_size = int(
        os.environ.get(
            "BATCH_SIZE",
            "0"
        )
    )

    # -------------------------------------------------------------------------
    # Device
    # -------------------------------------------------------------------------

    (
        device_type,
        device_label,
        fp16_enabled,
        detected_batch_size
    ) = detect_device()

    batch_size = (
        requested_batch_size
        if requested_batch_size > 0
        else detected_batch_size
    )

    # -------------------------------------------------------------------------
    # System information
    # -------------------------------------------------------------------------

    print("=" * 78, flush=True)

    print(
        "[System Init] Automatic device detection",
        flush=True
    )

    print(
        f"[System Init] Device      : {device_label}",
        flush=True
    )

    print(
        f"[System Init] CUDA        : "
        f"{torch.cuda.is_available()}",
        flush=True
    )

    print(
        f"[System Init] GPU count   : "
        f"{torch.cuda.device_count() if torch.cuda.is_available() else 0}",
        flush=True
    )

    print(
        f"[System Init] Batch size  : {batch_size}",
        flush=True
    )

    print(
        f"[System Init] FP16        : {fp16_enabled}",
        flush=True
    )

    print(
        f"[System Init] Samples     : {num_samples}",
        flush=True
    )

    print(
        f"[System Init] Epochs      : {NUM_EPOCHS}",
        flush=True
    )

    print("=" * 78, flush=True)

    # -------------------------------------------------------------------------
    # GPU information
    # -------------------------------------------------------------------------

    if torch.cuda.is_available():

        for i in range(torch.cuda.device_count()):

            props = torch.cuda.get_device_properties(i)

            total_gb = (
                props.total_memory
                / (1024 ** 3)
            )

            print(
                f"[GPU {i}] {props.name} | "
                f"VRAM: {total_gb:.2f} GB",
                flush=True
            )

    # =========================================================================
    # MODEL LOADING
    # =========================================================================

    phase_start = time.time()

    print(
        "\n[Model] Loading local pre-trained model...",
        flush=True
    )

    print(
        f"[Model] Path: {BASE_MODEL}",
        flush=True
    )

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        local_files_only=True
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL,
        local_files_only=True
    )

    print(
        f"[Timing] Model/tokenizer load: "
        f"{time.time() - phase_start:.1f}s",
        flush=True
    )

    # =========================================================================
    # DATASET PREPARATION
    # =========================================================================

    phase_start = time.time()

    dataset_split = prepare_squad_data(
        tokenizer,
        num_samples=num_samples
    )

    print(
        f"[Timing] Dataset preparation: "
        f"{time.time() - phase_start:.1f}s",
        flush=True
    )

    # =========================================================================
    # CALLBACK
    # =========================================================================

    logger_callback = TrainingLoggerCallback()

    # =========================================================================
    # TRAINING STEP CALCULATION
    # =========================================================================

    train_samples = len(
        dataset_split["train"]
    )

    steps_per_epoch = (
        train_samples + batch_size - 1
    ) // batch_size

    total_training_steps = (
        steps_per_epoch * NUM_EPOCHS
    )

    warmup_steps = max(
        1,
        int(total_training_steps * WARMUP_RATIO)
    )

    print(
        "\n[Training Configuration]",
        flush=True
    )

    print(
        f"Training samples       : {train_samples}",
        flush=True
    )

    print(
        f"Validation samples     : "
        f"{len(dataset_split['validation'])}",
        flush=True
    )

    print(
        f"Test samples           : "
        f"{len(dataset_split['test'])}",
        flush=True
    )

    print(
        f"Epochs                 : {NUM_EPOCHS}",
        flush=True
    )

    print(
        f"Batch size             : {batch_size}",
        flush=True
    )

    print(
        f"Steps per epoch        : {steps_per_epoch}",
        flush=True
    )

    print(
        f"Total training steps   : "
        f"{total_training_steps}",
        flush=True
    )

    print(
        f"Learning rate          : {LEARNING_RATE}",
        flush=True
    )

    print(
        f"Warmup ratio           : {WARMUP_RATIO}",
        flush=True
    )

    print(
        f"Warmup steps           : {warmup_steps}",
        flush=True
    )

    print(
        f"Input max length       : "
        f"{MAX_INPUT_LENGTH}",
        flush=True
    )

    print(
        f"Target max length      : "
        f"{MAX_TARGET_LENGTH}",
        flush=True
    )

    # =========================================================================
    # TRAINING ARGUMENTS
    # =========================================================================

    training_args = TrainingArguments(

        output_dir=str(SAVED_MODEL_DIR),

        # -------------------------------------------------------------
        # Training
        # -------------------------------------------------------------

        num_train_epochs=NUM_EPOCHS,

        per_device_train_batch_size=batch_size,

        per_device_eval_batch_size=batch_size,

        learning_rate=LEARNING_RATE,

        optim="adamw_torch",

        # Dynamic 10% warmup
        warmup_steps=warmup_steps,

        # -------------------------------------------------------------
        # Logging / Evaluation
        # -------------------------------------------------------------

        logging_steps=LOGGING_STEPS,

        eval_strategy="steps",

        eval_steps=EVAL_STEPS,

        # -------------------------------------------------------------
        # Saving
        # -------------------------------------------------------------

        save_strategy="no",

        # We save manually after training.
        # This keeps the output directory clean.
        # -------------------------------------------------------------

        # GPU
        fp16=fp16_enabled,

        # -------------------------------------------------------------
        # Reporting
        # -------------------------------------------------------------

        report_to="none",

        disable_tqdm=True,

        # -------------------------------------------------------------
        # Data loading
        # -------------------------------------------------------------

        dataloader_pin_memory=torch.cuda.is_available(),

        dataloader_num_workers=(
            2 if torch.cuda.is_available() else 0
        )
    )

    # =========================================================================
    # TRAINER
    # =========================================================================

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=dataset_split["train"],

        eval_dataset=dataset_split["validation"],

        callbacks=[logger_callback]
    )

    # =========================================================================
    # FINE-TUNING
    # =========================================================================

    print(
        "\n[2/4] Starting model fine-tuning process...",
        flush=True
    )

    train_start = time.time()

    trainer.train()

    # Synchronize CUDA before timing
    if torch.cuda.is_available():

        torch.cuda.synchronize()

    train_elapsed = (
        time.time() - train_start
    )

    train_mins, train_secs = divmod(
        int(train_elapsed),
        60
    )

    print(
        f"[Timing] Fine-tuning duration: "
        f"{train_mins}m {train_secs}s",
        flush=True
    )

    # =========================================================================
    # GPU MEMORY
    # =========================================================================

    if torch.cuda.is_available():

        for i in range(torch.cuda.device_count()):

            allocated = (
                torch.cuda.memory_allocated(i)
                / (1024 ** 3)
            )

            reserved = (
                torch.cuda.memory_reserved(i)
                / (1024 ** 3)
            )

            print(
                f"[GPU {i}] Memory - "
                f"Allocated: {allocated:.2f} GB | "
                f"Reserved: {reserved:.2f} GB",
                flush=True
            )

    # =========================================================================
    # HELD-OUT TEST SET
    # =========================================================================

    print(
        "\n[2.5/4] Evaluating fine-tuned model "
        "on held-out Test Set...",
        flush=True
    )

    test_start = time.time()

    test_results = trainer.evaluate(
        eval_dataset=dataset_split["test"],
        metric_key_prefix="test"
    )

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    test_elapsed = (
        time.time() - test_start
    )

    print(
        f"[Timing] Held-out test evaluation: "
        f"{test_elapsed:.1f}s",
        flush=True
    )

    test_loss = round(
        float(test_results.get("test_loss", 0.0)),
        4
    )

    print(
        f"[Test Set Metrics] "
        f"Held-Out Test Loss: {test_loss}",
        flush=True
    )

    # =========================================================================
    # SAVE TEST RESULTS
    # =========================================================================

    test_metrics = {

        "test_loss": test_loss,

        "eval_samples": len(
            dataset_split["test"]
        ),

        "num_total_samples": num_samples,

        "train_samples": len(
            dataset_split["train"]
        ),

        "validation_samples": len(
            dataset_split["validation"]
        ),

        "epochs": NUM_EPOCHS,

        "batch_size": batch_size,

        "learning_rate": LEARNING_RATE,

        "warmup_steps": warmup_steps,

        "max_input_length": MAX_INPUT_LENGTH,

        "max_target_length": MAX_TARGET_LENGTH,

        "model": BASE_MODEL,

        "device": device_label
    }

    with open(
        TEST_METRICS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            test_metrics,
            f,
            indent=4
        )

    print(
        f"[Artifact] Held-out test evaluation metrics "
        f"saved to '{TEST_METRICS_PATH.name}'",
        flush=True
    )

    # =========================================================================
    # SAVE MODEL
    # =========================================================================

    print(
        "\n[3/4] Saving model weights and tokenizer...",
        flush=True
    )

    SAVED_MODEL_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    model.save_pretrained(
        str(SAVED_MODEL_DIR)
    )

    tokenizer.save_pretrained(
        str(SAVED_MODEL_DIR)
    )

    print(
        f"[System Check] Model weights successfully "
        f"saved to '{SAVED_MODEL_DIR.name}'",
        flush=True
    )

    # =========================================================================
    # GENERATE TRAINING ARTIFACTS
    # =========================================================================

    print(
        "\n[4/4] Generating training artifacts...",
        flush=True
    )

    logger_callback.export_artifacts()

    # =========================================================================
    # FINAL SUMMARY
    # =========================================================================

    elapsed_time = (
        time.time() - start_time
    )

    mins, secs = divmod(
        int(elapsed_time),
        60
    )

    print("=" * 78, flush=True)

    print(
        f"[Execution Summary] "
        f"Device: {device_label} | "
        f"Samples: {num_samples} | "
        f"Epochs: {NUM_EPOCHS} | "
        f"Total Duration: {mins}m {secs}s | "
        f"Test Loss: {test_loss}",
        flush=True
    )

    print("-" * 78, flush=True)

    print(
        "[Artifacts]",
        flush=True
    )

    print(
        f"  Model       : {SAVED_MODEL_DIR}",
        flush=True
    )

    print(
        f"  Test Results: {TEST_METRICS_PATH}",
        flush=True
    )

    print(
        f"  Training Log: {CSV_LOG_PATH}",
        flush=True
    )

    print(
        f"  Loss Curve  : {PLOT_IMAGE_PATH}",
        flush=True
    )

    print("=" * 78, flush=True)

    print(
        "[Complete] Fine-tuning pipeline executed successfully!",
        flush=True
    )


# =============================================================================
# 9. TOP-LEVEL EXCEPTION HANDLER
# =============================================================================

if __name__ == "__main__":

    try:

        main()

    except KeyboardInterrupt:

        print(
            "\n[Interrupted] Fine-tuning execution "
            "was stopped by the user.",
            flush=True
        )

    except Exception as e:

        print(
            "\n[ERROR] Fine-tuning pipeline failed: "
            f"{type(e).__name__}: {e}",
            file=sys.stderr,
            flush=True
        )

        # In a normal Python script, return a non-zero exit code.
        # Avoid sys.exit() inside a Jupyter/IPython environment.
        if "IPython" not in sys.modules:

            sys.exit(1)

[System Init] Automatic device detection
[System Init] Device      : CUDA GPU (Tesla T4) x2
[System Init] CUDA        : True
[System Init] GPU count   : 2
[System Init] Batch size  : 16
[System Init] FP16        : True
[System Init] Samples     : 20000
[System Init] Epochs      : 3
[GPU 0] Tesla T4 | VRAM: 14.56 GB
[GPU 1] Tesla T4 | VRAM: 14.56 GB

[Model] Loading local pre-trained model...
[Model] Path: /kaggle/working/saved_pre_trained_model


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[Timing] Model/tokenizer load: 0.7s
[1/4] Loading SQuAD v1.1 (sample subset size: 20000)...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

...

[Dataset Split] Train: 16000 | Validation: 2000 | Test: 2000
[Timing] Dataset preparation: 13.4s

[Training Configuration]
Training samples       : 16000
Validation samples     : 2000
Test samples           : 2000
Epochs                 : 3
Batch size             : 16
Steps per epoch        : 1000
Total training steps   : 3000
Learning rate          : 5e-05
Warmup ratio           : 0.1
Warmup steps           : 300
Input max length       : 512
Target max length      : 64

[2/4] Starting model fine-tuning process...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[Training Step 50] Epoch 0.1 | Training Loss: 3.4865
{'loss': '3.486', 'grad_norm': '3.747', 'learning_rate': '8.167e-06', 'epoch': '0.1'}
[Validation Step 50] Epoch 0.1 | Validation Loss: 3.0804
{'eval_loss': '3.08', 'eval_runtime': '13.56', 'eval_samples_per_second': '147.4', 'eval_steps_per_second': '4.644', 'epoch': '0.1'}
[Training Step 100] Epoch 0.2 | Training Loss: 3.5148
{'loss': '3.515', 'grad_norm': '3.049', 'learning_rate': '1.65e-05', 'epoch': '0.2'}
[Validation Step 100] Epoch 0.2 | Validation Loss: 3.0656
{'eval_loss': '3.066', 'eval_runtime': '14.16', 'eval_samples_per_second': '141.2', 'eval_steps_per_second': '4.449', 'epoch': '0.2'}
[Training Step 150] Epoch 0.3 | Training Loss: 3.5009
{'loss': '3.501', 'grad_norm': '3.727', 'learning_rate': '2.483e-05', 'epoch': '0.3'}
[Validation Step 150] Epoch 0.3 | Validation Loss: 3.0491
{'eval_loss': '3.049', 'eval_runtime': '14.79', 'eval_samples_per_second': '135.2', 'eval_steps_per_second': '4.258', 'epoch': '0.3'}
[Trainin

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[System Check] Model weights successfully saved to 'saved_model'

[4/4] Generating training artifacts...
[Artifact] Unified training/validation metrics log saved to 'training_log.csv'
[Artifact] Step-level loss graph exported to 'loss_curve.png'
[Execution Summary] Device: CUDA GPU (Tesla T4) x2 | Samples: 20000 | Epochs: 3 | Total Duration: 23m 58s | Test Loss: 3.0279
------------------------------------------------------------------------------
[Artifacts]
  Model       : /kaggle/working/saved_model
  Test Results: /kaggle/working/test_results.json
  Training Log: /kaggle/working/training_log.csv
  Loss Curve  : /kaggle/working/loss_curve.png
[Complete] Fine-tuning pipeline executed successfully!


In [6]:
from datetime import datetime

print("=" * 70)
print("Training Run Timestamp")
print("Date & Time:", datetime.now().strftime("%d-%m-%Y %H:%M:%S"))
print("=" * 70)

Training Run Timestamp
Date & Time: 12-08-2026 04:32:50


In [7]:
import zipfile
from pathlib import Path

base_dir = Path("/kaggle/working")
zip_path = base_dir / "Group238_final_training_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:

    # Add complete saved_model folder
    model_dir = base_dir / "saved_model"

    for file in model_dir.rglob("*"):
        if file.is_file():
            z.write(
                file,
                arcname=file.relative_to(base_dir)
            )

    # Add generated artifacts
    for filename in [
        "training_log.csv",
        "loss_curve.png",
        "test_results.json"
    ]:

        file = base_dir / filename

        if file.exists():
            z.write(
                file,
                arcname=filename
            )

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")

Created: /kaggle/working/Group238_final_training_artifacts.zip
Size: 212.44 MB
